# Milestone 1 Exploration
## Data Import

This notebook is used for the following purpose:

- Import the data through duckdb for efficient loading
- EDA was kept in SQL to allow for scalability
- Our analysis was informed by exporation of:
    - The ratings proportion
    - The missingness of descriptor columns
    - The proportion of verified purchases
    - The distribution of helpful votes of reviews
    - The legnth of reviews grouped by ratings

In [1]:
from pathlib import Path
import duckdb
import requests
from tqdm import tqdm
import pandas as pd
import altair as alt

### Define the paths
The default option is set to "Appliances" category 

In [2]:
CATEGORY = "Appliances"
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
REVIEWS_URL = f"{BASE_URL}/review_categories/{CATEGORY}.jsonl.gz"
META_URL    = f"{BASE_URL}/meta_categories/meta_{CATEGORY}.jsonl.gz"
DATA_DIR = "../data/processed"
RAW_DIR = "../data/raw"

In [3]:
print(REVIEWS_URL)

https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Appliances.jsonl.gz


### Initialize an in-memory DB connection

In [4]:
c2 = duckdb.connect()

### Lets open the file online and see the first few lines

In [5]:
head_reviews = c2.execute(f"SELECT * FROM read_json_auto('{REVIEWS_URL}') LIMIT 5").df()
head_reviews

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Work great,work great. use a new one every month,[],B01N0TQ0OH,B01N0TQ0OH,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1519317108692,0,True
1,5.0,excellent product,Little on the thin side,[],B07DD2DMXB,B07DD37QPZ,AHWWLSPCJMALVHDDVSUGICL6RUCA,1664746863446,0,True
2,5.0,Happy customer!,"Quick delivery, fixed the issue!",[],B082W3Z9YK,B082W3Z9YK,AHZIJGKEWRTAEOZ673G5B3SNXEGQ,1607225435363,0,True
3,5.0,Amazing value,I wasn't sure whether these were worth it or n...,[],B078W2BJY8,B078W2BJY8,AFGUPTDFAWOHHL4LZDV27ERDNOYQ,1534104184306,0,True
4,5.0,Dryer parts,Easy to install got the product expected to re...,[],B08C9LPCQV,B08C9LPCQV,AELFJFAXQERUSMTXJQ6SYFFRDWMA,1620176603754,0,True


In [6]:
# executes right over the internet -- I just want five rows to preview so doesn't take long
head_meta = c2.execute(f"SELECT * FROM read_json_auto('{META_URL}') LIMIT 5").df()
head_meta

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Industrial & Scientific,"ROVSUN Ice Maker Machine Countertop, Make 44lb...",3.7,61,[【Quick Ice Making】This countertop ice machine...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Our Point of View on the Euhomy Ic...,ROVSUN,"[Appliances, Refrigerators, Freezers & Ice Mak...","{'Brand': '""ROVSUN""', 'Model Name': '""ICM-2005...",B08Z743RRD,None
1,Tools & Home Improvement,"HANSGO Egg Holder for Refrigerator, Deviled Eg...",4.2,75,"[Plastic, Practical Kitchen Storage - Our egg ...",[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': '10 Eggs Egg Holder for Refrigerato...,HANSGO,"[Appliances, Parts & Accessories, Refrigerator...","{'Manufacturer': '""HANSGO""', 'Part Number': '""...",B097BQDGHJ,None
2,Tools & Home Improvement,"Clothes Dryer Drum Slide, General Electric, Ho...",3.5,18,[],"[Brand new dryer drum slide, replaces General ...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],GE,"[Appliances, Parts & Accessories]","{'Manufacturer': '""RPI""', 'Part Number': '""WE1...",B00IN9AGAE,None
3,Tools & Home Improvement,154567702 Dishwasher Lower Wash Arm Assembly f...,4.5,26,[MODEL NUMBER:154567702 Dishwasher Lower Wash ...,[MODEL NUMBER:154567702 Dishwasher Lower Wash ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],folosem,"[Appliances, Parts & Accessories, Dryer Parts ...","{'Manufacturer': '""folosem""', 'Part Number': '...",B0C7K98JZS,None
4,Tools & Home Improvement,Whirlpool W10918546 Igniter,3.8,12,[This is a Genuine OEM Replacement Part.],[Whirlpool Igniter],25.07,[{'thumb': 'https://m.media-amazon.com/images/...,[],Whirlpool,"[Appliances, Parts & Accessories]","{'Manufacturer': '""Whirlpool""', 'Part Number':...",B07QZHQTVJ,None


### Download and convert to parquet on the fly

So lets download and covert to parquet on the fly (remember, we pay conversion tax once and reuse for all consequent queries -- whether in duckdb or pandas ...). 

Even if we are performing parallel processing, we need full download here, so it will take time, depending on your connection + conversion overhead.

- Downloading only 20k (`LIMIT 20000`) for a quick inspection. 
- It take a few seconds to download (will depend on your network connection)  

In [7]:
c2.execute(
    f"""
      COPY (SELECT * FROM read_json_auto('{REVIEWS_URL}')  LIMIT 20000)
      TO '{RAW_DIR}/reviews_raw.parquet'
      (FORMAT PARQUET, COMPRESSION ZSTD)
  """
)

In [8]:
c2.execute(
    f"""
      COPY (SELECT * FROM read_json_auto('{META_URL}') LIMIT 20000)
      TO '{RAW_DIR}/meta_raw.parquet'
      (FORMAT PARQUET, COMPRESSION ZSTD)
  """
)

### Merging the two files by joining on `parent_asin`

In [9]:
c2.execute(
    f"""
    COPY (
        SELECT 
        r.*, m.title AS product_title, m.features,
        m.description, m.categories, m.details,
        m.price,
        m.average_rating, m.main_category, m.store,
        FROM read_parquet('{RAW_DIR}/reviews_raw.parquet') r
        LEFT JOIN read_parquet('{RAW_DIR}/meta_raw.parquet') m USING (parent_asin)
    )
    TO '{DATA_DIR}/merged.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
"""
)

## EDA

### Description

In [10]:
c2.execute(
    f"""
    DESCRIBE SELECT *
    FROM read_parquet('{DATA_DIR}/merged.parquet')
"""
).df()

,column_name,column_type,null,key,default,extra
0,rating,DOUBLE,YES,None,None,None
1,title,VARCHAR,YES,None,None,None
2,text,VARCHAR,YES,None,None,None
3,images,"STRUCT(small_image_url VARCHAR, medium_image_u...",YES,None,None,None
4,asin,VARCHAR,YES,None,None,None
5,parent_asin,VARCHAR,YES,None,None,None
6,user_id,VARCHAR,YES,None,None,None
7,timestamp,BIGINT,YES,None,None,None
8,helpful_vote,BIGINT,YES,None,None,None
9,verified_purchase,BOOLEAN,YES,None,None,None


In [11]:
c2.execute(
    f"""
    SELECT *
    FROM read_parquet('{DATA_DIR}/merged.parquet')
    LIMIT 10
"""
).df()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,product_title,features,description,categories,details,price,average_rating,main_category,store
0,5.0,Just what we needed.,Just what we needed.,[],B07W42P978,B07W42P978,AE6YJEKU5YTRN7S6AOWQIQITAA4Q,1658966778279,0,True,WD12X10327 Rack Roller and stud assembly Kit (...,[【PARTS NUMBER】The WD12X10327 dishwasher top r...,[],"[Appliances, Parts & Accessories, Dishwasher P...","{'Brand Name': '""AMI PARTS""', 'Model Info': '""...",8.99,4.6,Appliances,AMI PARTS
1,5.0,Excellent Cold Brew Filter,"This filter is tall, has a very fine mesh and ...",[],B083Q6Y54F,B083Q6Y54F,AEOLBUYSTR77TPOFRYZCMBG67EDQ,1655952233309,0,True,G.a HOMEFAVOR Cold Brew Coffee Infuser 64oz (2...,[COMPLETE COLD BREW SYSTEM: The Cold Brew Kit ...,[],"[Small Appliance Parts & Accessories, Coffee &...","{'Package Dimensions': '""8.46 x 3.43 x 3.39 in...",NaN,4.5,Amazon Home,G.a HOMEFAVOR
2,1.0,Roller bearing too tight,I have installed hundreds of these rollers the...,[],B07TVL7F1M,B0BLZQYHPT,AEH5FSE32L2LIC32FLO7RC37ZPEQ,1618592606367,0,True,349241T Dryer Drum Roller Kit Whirlpool Kenmor...,[💡 Drum Roller 349241t - An equivalent for par...,[],"[Appliances, Parts & Accessories, Dryer Parts ...","{'Manufacturer': '""PartsBroz""', 'Part Number':...",7.95,4.3,Industrial & Scientific,PartsBroz
3,5.0,Works well,Same as OEM product. Was easy and quick to ins...,[],B01AQHSNSM,B01AQHSNSM,AF5VV27MNCDLOE2UHUSE5ME6HD7Q,1612840366382,0,True,SAMSUNG DA97-11092B Genuine OEM Ice Maker Asse...,[The Samsung DA97-11092B Ice Maker Assembly is...,[This high quality Genuine OEM Samsung Ice Mak...,"[Appliances, Parts & Accessories, Refrigerator...","{'Manufacturer': '""J&J International Inc.""', '...",138.00,4.6,Tools & Home Improvement,SAMSUNG
4,5.0,Getting very fast shipping!,Can't live without them!,[],B077H7V2FZ,B0BWCL9WCH,AHB2N4CJTGWOXRPFV4RLX6EOXDAQ,1563777103442,0,True,PARTY BARGAINS 600 Paper Coffee Filters - Whit...,[✅ HIGH-QUALITY COFFEE FILTERS - Made of premi...,[PARTY BARGAINS Disposable Coffee Filters | Si...,"[Small Appliance Parts & Accessories, Coffee &...","{'Package Dimensions': '""9.02 x 6.5 x 5.55 inc...",28.99,4.5,Amazon Home,PARTY BARGAINS
5,5.0,Good product,Worked as expected. Typical fast delivery.,[],B09XWHZNP7,B0BJKNMKRP,AHOPYBWVAJLXPCKTQTPKXWLDHKKQ,1678751892111,0,True,HERISUN 100 Disposable Coffee Filters for Sing...,[【Good Fits in All Refillable Pods】K cup paper...,[],"[Small Appliance Parts & Accessories, Coffee &...","{'Product Dimensions': '""4.7 x 2.5 x 2.5 inche...",7.99,4.4,Amazon Home,HERISUN
6,4.0,Great to have plenty of ice again!,"Our ice maker on our fridge broke, Because of...",[],B07T64VJTH,B0BLC6GRLX,AHHG5GS2XOMFHM5KHGYD3M67D7DQ,1588089470949,6,True,VIVOHOME Electric Portable Compact Countertop ...,[LIGHTWEIGHT & PORTABLE - Lightweight feature ...,[],"[Appliances, Refrigerators, Freezers & Ice Mak...","{'Brand': '""VIVOHOME""', 'Model Name': '""Electr...",139.99,4.4,Industrial & Scientific,VIVOHOME
7,3.0,My experience,If ONE CUBE sits on the sensor for ANY reason ...,[],B01AYKYBAA,B01AYKYBAA,AFLWQYLJQ5OX3YJEC4ESRKTGTJVA,1532629139931,0,True,DELLA Compact Portable (Red) Ice Maker Machine...,[Modern compressor refrigeration technology fo...,"[Compact & portable- perfect for your RV, boat...","[Appliances, Refrigerators, Freezers & Ice Mak...","{'Brand': '""Della""', 'Model Name': '""FBA_048-G...",NaN,4.0,Amazon Home,Della
8,5.0,Received in a timely manner and was exact repl...,Received in a timely manner and was exact repl...,[],B07W1HQQ77,B07W1HQQ77,AE5WO2DIUMOK3VTVSIGLCYLCN46A,1571336700182,0,True,"Dryer Belt,341241 Dryer Drum Belt Compatible w...",[],[],"[Appliances, Parts & Accessories, Dryer Parts ...","{'Part Number': '""K0809B01""', 'Item Weight': '...",NaN,5.0,Tools & Home Improvement,AIEVE
9,5.0,Good price and worked as expected,They were reasonably priced and easy to instal...,[],B08DCM4TXN,B0C5F65FMS,AGA4GLOQJELMEFI76DMRPTNBFM3Q,1626773458907,0,Tru

### Ratings Proportion

In [12]:
ratings = c2.execute(
    f"""
    SELECT
        rating,
        COUNT(*) AS cnt,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM read_parquet('{DATA_DIR}/merged.parquet')
    GROUP BY 1
    ORDER BY 1
"""
).df()

ratings

,rating,cnt,pct
0,1.0,1484,7.42
1,2.0,624,3.12
2,3.0,1139,5.70
3,4.0,2304,11.52
4,5.0,14449,72.25


In [13]:
alt.Chart(ratings).mark_bar().encode(
    x=alt.X("rating:N", title="Rating Bins"),
    y=alt.Y(
        "pct:Q",
        scale=alt.Scale(domain=[0, 100]),
        title="Proportion",
    ),
    tooltip=[alt.Tooltip("ratings:N"), alt.Tooltip("pct:Q")],
).properties(
    width=600, height=400, title="Proportion of Ratings"
)

alt.Chart(...)

This is from the user reviews dataset, where there are no missing values. The majority if ratings are at 5.0. However, this is not representative of the metadata, where the majority of purchases do not result in a review.

### Missingness

Screening for missingingness in descriptor columns to see if this informs which columns to prioritize as an information request. 

First, explored what missingness could look like in these lists / strings.

In [14]:
# NOTE: SQL code was taken from chatgpt5

c2.execute(
    f"""
SELECT DISTINCT CAST(features AS VARCHAR) AS val, 'features' AS col
FROM read_parquet('{DATA_DIR}/merged.parquet')
WHERE LENGTH(TRIM(CAST(features AS VARCHAR))) < 3

UNION

SELECT DISTINCT CAST(description AS VARCHAR) AS val, 'description' AS col
FROM read_parquet('{DATA_DIR}/merged.parquet')
WHERE LENGTH(TRIM(CAST(description AS VARCHAR))) < 3

UNION

SELECT DISTINCT CAST(categories AS VARCHAR) AS val, 'categories' AS col
FROM read_parquet('{DATA_DIR}/merged.parquet')
WHERE LENGTH(TRIM(CAST(categories AS VARCHAR))) < 3

ORDER BY col, val
"""
).df()

,val,col
0,[],categories
1,[],description
2,[],features


In [15]:
# NOTE: SQL code was taken from chatgpt5

missing_df = c2.execute(
    f"""
SELECT
    AVG(CASE WHEN categories IS NULL OR LENGTH(categories) = 0 THEN 1 ELSE 0 END) AS prop_categories_missing,
    AVG(CASE WHEN description IS NULL OR LENGTH(description) = 0 THEN 1 ELSE 0 END) AS prop_description_missing,
    AVG(CASE WHEN features IS NULL OR LENGTH(features) = 0 THEN 1 ELSE 0 END) AS prop_features_missing,

    AVG(CASE WHEN text IS NULL OR TRIM(text) = '' THEN 1 ELSE 0 END) AS prop_text_missing,
    AVG(CASE WHEN store IS NULL OR TRIM(store) = '' THEN 1 ELSE 0 END) AS prop_store_missing,
    AVG(CASE WHEN product_title IS NULL OR TRIM(product_title) = '' THEN 1 ELSE 0 END) AS prop_title_missing,

    AVG(CASE WHEN price IS NULL OR isnan(price) THEN 1 ELSE 0 END) AS prop_price_missing,
    AVG(CASE WHEN average_rating IS NULL OR isnan(average_rating) THEN 1 ELSE 0 END) AS prop_avg_rating_missing,
    AVG(CASE WHEN rating IS NULL OR isnan(rating) THEN 1 ELSE 0 END) AS prop_rating_missing,

    FROM read_parquet('{DATA_DIR}/merged.parquet')
"""
).df()

plot_df = missing_df.melt(var_name="column", value_name = "missingness")

print(plot_df)

alt.Chart(plot_df).mark_bar().encode(
    x=alt.X("column:N", sort="-y", title="Descriptor Columns"),
    y=alt.Y(
        "missingness:Q",
        scale=alt.Scale(domain=[0, 1]),
        title="Missingness Proportions",
    ),
    tooltip=[
        alt.Tooltip("column:N"),
        alt.Tooltip("missingness:Q")
    ]
).properties(width=600, height=400, title="Proportion of Descriptor Columns that are Missing")

                     column  missingness
0   prop_categories_missing      0.63860
1  prop_description_missing      0.80940
2     prop_features_missing      0.65290
3         prop_text_missing      0.00005
4        prop_store_missing      0.63635
5        prop_title_missing      0.63585
6        prop_price_missing      0.70325
7   prop_avg_rating_missing      0.63585
8       prop_rating_missing      0.00000


alt.Chart(...)

There's a high degree of missingness among descriptor columns. This doesn't necessary mean we should focus on any one column, but that they all may need to be included to pick up as much information as possible based on the sparsity of data.

This gives a better view of the lack of ratings for all purchases, compared to early analysis. Also, text and ratings is from the user review df, and so it seem every review has text and a rating. 

### Verified Purchases

Only a small proportion are not verified purchases. So they could still be included, but should be kept as a factor in case we want to remove the FALSE items.

In [16]:
c2.execute(
    f"""
SELECT
    DISTINCT verified_purchase,
    COUNT(*) AS cnt,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM read_parquet('{DATA_DIR}/merged.parquet')
GROUP BY 1
ORDER BY 1
"""
).df()

,verified_purchase,cnt,pct
0,False,1501,7.51
1,True,18499,92.50


### Helpful Votes of Reviews

Majority of reviews have no votes as being helpful; however, this could be a good column of sorting reviews, if we could to just show 1 candidate review on the web app

In [17]:
c2.execute(
    f"""
SELECT
    helpful_vote,
    COUNT(*) AS countS
FROM read_parquet('{DATA_DIR}/merged.parquet')
GROUP BY helpful_vote
ORDER BY helpful_vote
"""
).df()

,helpful_vote,countS
0,0,15681
1,1,2139
2,2,696
3,3,403
4,4,189
...,...,...
88,216,1
89,268,1
90,425,1
91,593,1


### Average Review Length

In [18]:
# NOTE: SQL code taken from chatGPT5

plot_df = c2.execute(
    f"""
SELECT
    rating,
    MEDIAN(LENGTH(text)) AS median_text_len,
    COUNT(*) AS n
FROM read_parquet('{DATA_DIR}/merged.parquet')
WHERE text IS NOT NULL
  AND TRIM(text) <> ''
  AND rating IS NOT NULL
GROUP BY rating
ORDER BY rating
"""
).df()

In [19]:
import altair as alt

alt.Chart(plot_df).mark_bar().encode(
    x=alt.X("rating:O", title="Rating"),
    y=alt.Y("median_text_len:Q", title="Average Text Length"),
    tooltip=["rating", "median_text_len", "n"],
).properties(width=500, height=300, title="Average Review Text Length by Rating")

alt.Chart(...)

This is informative because the negative reviews tend to be more verbose, so this may affect the corpus and the probability of negative reviews comapred to positive review. 